# GameTheory 3b — Chambres, murs et chemins : la géométrie ordinale des jeux 2×2

> **Deuxième volet de la chaîne 03** — après [GameTheory-03-Topology2x2-Python](GameTheory-03-Topology2x2-Python.ipynb) (les 576 jeux stricts et la grammaire des swaps), avant [GameTheory-03c-Le-Joueur-LLM-Python](GameTheory-03c-Le-Joueur-LLM-Python.ipynb) (la table périodique appliquée à un LLM), [GameTheory-03d-Deux-Especes-de-Fleches-Python](GameTheory-03d-Deux-Especes-de-Fleches-Python.ipynb) (quand un swap préserve les équilibres) et [GameTheory-03e-Meta-Actions-Tarifees-Python](GameTheory-03e-Meta-Actions-Tarifees-Python.ipynb) (le capstone).

Ce carnet décrit **l'espace** dans lequel les transformations de GT-3 se déplacent, puis **sa métrique** : les strates de l'espace des paiements et leur incidence (§1-§4), puis la distance de swap entre deux jeux et les chemins qui la réalisent (§5-§10). Les deux moitiés se lisent dans cet ordre — une distance se mesure en murs traversés, et les murs sont ce que la première moitié construit.

**La thèse, en une phrase** : un jeu à égalités n'est pas un jeu « plus flou » ou « plus général » qu'un jeu strict — c'est un objet **de dimension inférieure**, un mur où des chambres se touchent ; et la distance de swap entre deux jeux stricts est la somme des longueurs de Coxeter de leurs deux côtés, mesurée par le nombre de murs traversés.

La lecture géométrique vient de la formalisation de **Bruns & Kimmich** (rapportée, cf. §Sources) :

- les jeux **stricts** — aucune égalité de paiements — sont les **chambres** : les composantes ouvertes de l'espace des paiements ;
- les jeux **à égalités** sont les **murs** : des strates de **codimension ≥ 1** sur lesquelles plusieurs chambres se raccordent ;
- une transformation élémentaire n'est pas un saut abstrait d'un jeu à un autre : elle **traverse une face**.

```
chambre  →  mur  →  chambre voisine
```

Un « objet botanique » — un jeu à égalités depuis lequel plusieurs prolongements sont possibles — n'est donc **pas** une définition assouplie : c'est **habiter le bord**. La différence entre les deux notions est la différence entre *relâcher un axiome* et *se placer sur une strate de codimension 1*.

**Objectifs d'apprentissage** — à la fin de ce carnet, vous savez :

1. énumérer les strates de l'espace des paiements et les classer par codimension (75 ordres faibles par joueur, 5 625 jeux à rangs) ;
2. mesurer l'incidence chambre-mur et lire le graphe des chambres (connexité, diamètre, distribution des distances) ;
3. calculer une distance de swap par parcours en largeur, et énumérer les plus courts chemins entre deux jeux ;
4. vérifier le théorème de décomposition $d(G,H) = \ell(r_G,r_H) + \ell(c_G,c_H)$ par énumération exhaustive, et distinguer un chemin **valide** d'un chemin **minimal** — le premier se certifie, le second se mesure.

**Prérequis** : [GameTheory-03-Topology2x2-Python](GameTheory-03-Topology2x2-Python.ipynb) — les 576 jeux stricts et les six générateurs $R_{12}/R_{23}/R_{34}/C_{12}/C_{23}/C_{34}$. Python 3.10+, aucune dépendance externe.

**Durée estimée** : 1 h 30 de lecture active, plus les six exercices.

**Plan** : §1 les strates classées par codimension · §2 l'incidence chambre-mur et le graphe des chambres · §3 traverser un mur, puis relire la grammaire des swaps en longueurs de Coxeter · §4 *make tie* / *break tie* et trois archétypes · §5 l'espace des 576 jeux et ses générateurs · §6 la distance d'un jeu à l'autre — BFS depuis le Dilemme, tableau des distances aux archétypes, les trois cousins à distance 2 · §7 la distribution des distances — sphères emboîtées, symétrie parfaite, convolution des vecteurs de Mahler · §8 le théorème de décomposition — la distance est la somme des longueurs de Coxeter des deux côtés, vérifié par énumération exhaustive sur les 331 776 paires · §9 les multiplicités des plus courts chemins — de 2 (commutation) à 236 544 (l'anti-Dilemme) · §10 générateur contre certificat — le miroir Python du vérificateur Lean, valide ≠ minimal, puis le module `Swaps/Basic.lean` lui-même, cité aux lignes mesurées · **Exercices** (6) — énumérer les murs par codimension, exhiber un triplet chambre-mur-chambre, make tie / break tie vers les archétypes, puis distance par sphères, diamètre depuis une autre source et certificat d'un chemin fourni.

**Conventions** : un jeu 2×2 ordinal = deux tables de paiements (Ligne, Colonne), chacune un 4-uplet $(r_{11}, r_{12}, r_{21}, r_{22})$ de valeurs ; un jeu strict = chaque table est une permutation de $\{1,2,3,4\}$.


## 0. Configuration

Notebook **pur standard library** (`itertools`, `collections`) — aucune dépendance. Comme GT-21, tout ce qui est affirmé ici est **calculé dans ce notebook** ; ce qui vient de la littérature sans dérivation locale est **rapporté comme dette** (§Sources).

In [1]:
# GameTheory 3b : chambres, murs et chemins -- pur standard library
from itertools import product, permutations
from collections import Counter, deque, defaultdict

print("GameTheory 3b : chambres, murs et chemins")
print("Representation : jeu = (table_ligne, table_colonne), tables = ordres faibles canoniques")


GameTheory 3b : chambres, murs et chemins
Representation : jeu = (table_ligne, table_colonne), tables = ordres faibles canoniques


## 1. L'espace des paiements et ses strates

### 1.1 Ordres faibles : la clé canonique

L'espace des paiements d'un joueur est $\mathbb{R}^4$ (une coordonnée par case de sa table 2×2). Ce qui compte pour un jeu ordinal n'est pas la valeur numérique d'une coordonnée mais son **rang** relativement aux trois autres. Un point de cet espace code donc un **ordre faible** (*weak order*) sur les 4 cases : soit un ordre total (les 4 valeurs distinctes — une chambre), soit un ordre avec des ex æquo (un mur).

La représentation canonique : un 4-uplet $t$ où $t_i = 1 + (\text{nombre de valeurs distinctes strictement inférieures à } t_i)$. Ainsi $(2,2,4,5)$, $(1,1,3,4)$ et $(7,7,8,9)$ codent le même ordre faible — deux cases liées en bas, deux cases liées au-dessus — et partagent la clé canonique $(1,1,2,3)$.

Le discret $\{1,2,3,4\}$ de GT-3/GT-21 est le cas particulier sans ex æquo : les 24 permutations de $(1,2,3,4)$.

In [2]:
# === Section 1.1 : enumeration des ordres faibles sur 4 cases ===

def canonique(t):
    """Cle canonique d'un 4-uplet : relabelage croissant des valeurs distinctes."""
    vals = sorted(set(t))
    return tuple(vals.index(v) + 1 for v in t)

# Enumeration : tous les 4-uplets de {1..4} (256), projetes sur leur cle canonique
ordres_faibles = sorted({canonique(t) for t in product(range(1, 5), repeat=4)})
print("Ordres faibles sur 4 cases :", len(ordres_faibles))
print("  stricts (chambres potentielles) :", sum(1 for t in ordres_faibles if len(set(t)) == 4))
print("  avec ex aequo (murs potentiels)  :", sum(1 for t in ordres_faibles if len(set(t)) < 4))


Ordres faibles sur 4 cases : 75
  stricts (chambres potentielles) : 24
  avec ex aequo (murs potentiels)  : 51


**Lecture de la sortie committée** : **75 ordres faibles** — c'est le *nombre de Fubini* $a(4)$, le compte des partitions ordonnées d'un ensemble à 4 éléments. Ils se partagent en **24 ordres totaux** (les permutations — les chambres au sens strict, inchangées depuis GT-3) et **51 ordres avec ex æquo** (les murs, par joueur).

Ce simple compte situe le changement de regard : le discret de GT-3 ($24^2 = 576$ jeux stricts) vivait dans un espace où les égalités étaient **exclues par construction**. En admettant les ex æquo dans la représentation, l'espace passe de $576$ à $75^2 = 5625$ jeux à rangs — et ces nouveaux points ne sont pas des « jeux imprécis » : ce sont des **strates**. Reste à leur donner une dimension.

### 1.2 Codimension : compter des égalités *indépendantes*

La codimension d'une strate est le nombre d'**équations indépendantes** qui la définissent — la dimension perdue. Pour un ordre faible, ce n'est pas le nombre de *paires* égales mais le **complément du nombre de blocs** :

$$\text{codim}(t) = 4 - \#\text{blocs}(t).$$

Un bloc de taille 3 (triple égalité) porte **3 paires** égales mais seulement **2 équations indépendantes** ($x_a = x_b$ et $x_b = x_c$ suffisent, la troisième s'en déduit). Confondre paires et équations est l'erreur classique — le code suivant les compte séparément pour montrer qu'elles divergent.

In [3]:
# === Section 1.2 : codimension vraie vs nombre de paires egales ===

def nb_blocs(t):
    return len(set(t))

def nb_paires_egales(t):
    return sum(m * (m - 1) // 2 for m in Counter(t).values())

def codim(t):
    return 4 - nb_blocs(t)

print("Type de blocs           | codim | paires | exemples")
print("-" * 64)
par_type = defaultdict(list)
for t in ordres_faibles:
    signature = tuple(sorted(Counter(t).values(), reverse=True))
    par_type[signature].append(t)
for sig in sorted(par_type, reverse=True):
    ex = par_type[sig]
    paires_moy = sum(nb_paires_egales(t) for t in ex) // len(ex)
    exemples = str(ex[0]) + (", " + str(ex[1]) if len(ex) > 1 else "")
    print(f"blocs {str(sig):10s} ({len(ex):2d} ordres) |   {4 - len(sig)}   |   {paires_moy}    | {exemples}")
print()
dist_codim = Counter(codim(t) for t in ordres_faibles)
print("Distribution par codimension (par joueur) :", dict(sorted(dist_codim.items())))
print("Verification de la somme : 24 + 36 + 14 + 1 =", sum(dist_codim.values()), "= 75 ordres faibles")


Type de blocs           | codim | paires | exemples
----------------------------------------------------------------
blocs (4,)       ( 1 ordres) |   3   |   6    | (1, 1, 1, 1)
blocs (3, 1)     ( 8 ordres) |   2   |   3    | (1, 1, 1, 2), (1, 1, 2, 1)
blocs (2, 2)     ( 6 ordres) |   2   |   2    | (1, 1, 2, 2), (1, 2, 1, 2)
blocs (2, 1, 1)  (36 ordres) |   1   |   1    | (1, 1, 2, 3), (1, 1, 3, 2)
blocs (1, 1, 1, 1) (24 ordres) |   0   |   0    | (1, 2, 3, 4), (1, 2, 4, 3)

Distribution par codimension (par joueur) : {0: 24, 1: 36, 2: 14, 3: 1}
Verification de la somme : 24 + 36 + 14 + 1 = 75 = 75 ordres faibles


**Lecture de la sortie committée** : la taxonomy par taille de blocs croise les deux mesures.

| blocs | codim | paires égales | compte |
|---|---|---|---|
| $(1,1,1,1)$ | 0 | 0 | 24 — **les chambres** |
| $(2,1,1)$ | 1 | 1 | 36 — **les murs simples** |
| $(2,2)$ | 2 | 2 | 6 |
| $(3,1)$ | **2** | **3** | 8 |
| $(4)$ | **3** | **6** | 1 |

Les deux lignes en **gras** portent la leçon : le triple ex æquo (les 8 cas $(3,1)$) **diverge** — 3 paires mais codimension 2 — et le quadruple ex æquo porte 6 paires pour codimension 3. La codimension est l'auche correcte : elle mesure la **dimension perdue**, pas le nombre de coïncidences.

L'image botanique se précise : les murs de codimension 1 sont des **cloisons** (36 par joueur), ceux de codimension 2 des **arêtes** où les cloisons se rencontrent (6+8 = 14), celui de codimension 3 un **sommet** (1). Un objet botanique n'est pas « un jeu imprécis » : c'est un point de l'espace des paiements situé **sur** cette architecture.

### 1.3 Les 5 625 jeux à rangs, par codimension

Un jeu complet = une paire d'ordres faibles (Ligne, Colonne). La codimension du jeu est la somme des deux : un mur de Ligne dans une chambre de Colonne est un mur de codim 1 ; deux murs simultanés donnent codim 2 ; etc.

In [4]:
# === Section 1.3 : distribution des jeux par codimension totale ===

jeux = [(r, c) for r in ordres_faibles for c in ordres_faibles]
dist_jeux = Counter(codim(r) + codim(c) for r, c in jeux)
print("Jeux a rangs (paires d'ordres faibles) :", len(jeux))
print()
print("codim |  jeux  | remarque")
print("-" * 56)
remarques = {0: "les 576 chambres strictes (univers GT-3 / GT-21)",
             1: "murs simples : un seul ex aequo, d'un seul cote",
             2: "murs doubles : deux ex aequo (eventuellement croises)",
             6: "les deux tables entierement plates (jeu unique)"}
for k in sorted(dist_jeux):
    print(f"  {k}   | {dist_jeux[k]:5d}  | {remarques.get(k, '')}")
print()
print("Total :", sum(dist_jeux.values()), "= 75 x 75 attendu")
print("Chambres strictes :", dist_jeux[0], "-- coherentes avec GT-3 (576) et GT-21")


Jeux a rangs (paires d'ordres faibles) : 5625

codim |  jeux  | remarque
--------------------------------------------------------
  0   |   576  | les 576 chambres strictes (univers GT-3 / GT-21)
  1   |  1728  | murs simples : un seul ex aequo, d'un seul cote
  2   |  1968  | murs doubles : deux ex aequo (eventuellement croises)
  3   |  1056  | 
  4   |   268  | 
  5   |    28  | 
  6   |     1  | les deux tables entierement plates (jeu unique)

Total : 5625 = 75 x 75 attendu
Chambres strictes : 576 -- coherentes avec GT-3 (576) et GT-21


**Lecture de la sortie committée** : l'univers des jeux à rangs compte **5 625** états. Les **576 chambres strictes** — tout l'univers de GT-3 et de GT-21 — n'en occupent que **10,2 %**. Les **1 728 jeux de codimension 1** sont les murs simples ; **1 968** de codim 2 ; le reste s'étage jusqu'au jeu totalement plat (codim 6), unique, où chaque joueur est indifférent entre les quatre cases.

Autrement dit : le discret dans lequel la série a travaillé jusqu'ici était le **squelette ouvert** d'un édifice dont les murs, arêtes et sommets — 89,8 % des états à rangs — n'avaient pas de nom. La suite leur donne un rôle structurel : ce sont eux qui **raccordent** les chambres.

## 2. L'incidence chambre-mur

### 2.1 Chaque mur simple sépare exactement deux chambres

Un mur de codimension 1 (un seul ex æquo, d'un seul côté) est une cloison : la briser dans un sens ou dans l'autre donne **exactement deux** chambres. Réciproquement, depuis une chambre, quels murs touche-t-on ? Pas tous : pour que deux cases puissent **se rencontrer**, il faut qu'aucune autre valeur ne soit strictement comprise entre les leurs — sinon le mouvement croiserait d'autres murs d'abord. Les facettes d'une chambre correspondent aux paires de **valeurs adjacentes** dans son ordre.

Le code suivant mesure l'incidence dans les deux directions.

In [5]:
# === Section 2.1 : incidence double-face chambre <-> mur ===

chambres = [t for t in ordres_faibles if codim(t) == 0]      # 24 permutations
murs_simples = [t for t in ordres_faibles if codim(t) == 1]  # 36

def briser_le_tie(t):
    """Les deux chambres strictes adjacentes au mur simple t (cote de t seul)."""
    c = Counter(t)
    v_tie = [k for k, m in c.items() if m == 2][0]
    i, j = [p for p in range(4) if t[p] == v_tie]           # les deux cases liees
    below = sorted([p for p in range(4) if t[p] < v_tie], key=lambda p: t[p])
    above = sorted([p for p in range(4) if t[p] > v_tie], key=lambda p: t[p])
    # chambre A : i passe devant j ; chambre B : j devant i -- au NIVEAU du tie
    def rangs(ordre_positions):
        rk = [0] * 4
        for k, p in enumerate(ordre_positions):
            rk[p] = k + 1
        return tuple(rk)
    A = rangs(below + [i, j] + above)
    B = rangs(below + [j, i] + above)
    return A, B

# mur -> chambres
incidences_mur = {t: briser_le_tie(t) for t in murs_simples}
toutes_distinctes = all(a != b for a, b in incidences_mur.values())
print("Murs simples par cote :", len(murs_simples))
print("Chaque mur touche exactement 2 chambres distinctes :", toutes_distinctes)

# chambre -> murs (facettes)
facettes = defaultdict(list)
for t, (a, b) in incidences_mur.items():
    facettes[a].append(t)
    facettes[b].append(t)
print("Facettes par chambre  :", dict(Counter(len(v) for v in facettes.values())),
      "(3 attendu : les 3 paires de valeurs adjacentes)")


Murs simples par cote : 36
Chaque mur touche exactement 2 chambres distinctes : True
Facettes par chambre  : {3: 24} (3 attendu : les 3 paires de valeurs adjacentes)


**Lecture de la sortie committée** : la double incidence est **exacte**.

- **Mur → chambres** : chacun des 36 murs simples d'un côté touche **exactement 2 chambres** distinctes — les deux résolutions de son ex æquo. Une cloison n'a jamais trois pièces de chaque côté.
- **Chambre → murs** : chacune des 24 chambres a **exactement 3 facettes** — les 3 paires de valeurs *adjacentes* de son ordre. Une paire non adjacente (les valeurs 1 et 3, avec 2 entre elles) ne définit **pas** une facette : pour amener 1 et 3 à l'égalité, il faut d'abord traverser le mur où 1 rencontre 2.

C'est la structure d'**arrangement d'hyperplans** de l'espace des paiements (les murs étant les hyperplans $x_i = x_j$) — calculée ici sans en invoquer la théorie : les comptes parlent d'eux-mêmes. Au niveau du jeu complet (deux côtés), chaque chambre a donc $3 + 3 = 6$ facettes, et le nombre de murs simples du jeu vaut $2 \times 36 \times 24 = 1728$.

**Remarque de parité avec GT-21** : le théorème « le mur ne compte que s'il est habité » de GT-21 portait sur les *meilleures réponses* (quelles stratégies habitent la colonne traversée) ; ici l'habitation est *géométrique* — quelles chambres touchent le mur. Les deux lectures se complètent : GT-21 dit quand un swap **change** les équilibres, la lecture géométrique dit **où** le swap se déplace.

### 2.2 Le graphe des chambres : connexité et diamètre

Déclarer que les murs « raccordent » les chambres donne un graphe : les sommets sont les 576 chambres, une arête par mur simple traversé. La bonne nouvelle pour la grammaire des swaps : **ce graphe est connexe** — n'importe quel jeu strict est atteignable depuis n'importe quel autre en traversant des murs un à un.

In [6]:
# === Section 2.2 : BFS sur le graphe des chambres ===

def swap_valeurs_adjacentes(t, k):
    """Echange les cases portant les valeurs k et k+1 (traversee de la facette k<->k+1)."""
    pk, pk1 = t.index(k), t.index(k + 1)
    l = list(t)
    l[pk], l[pk1] = l[pk1], l[pk]
    return tuple(l)

adjacences = {t: [swap_valeurs_adjacentes(t, k) for k in (1, 2, 3)] for t in chambres}

# BFS cote 1 joueur, depuis (1,2,3,4)
depart = (1, 2, 3, 4)
dist = {depart: 0}
q = deque([depart])
while q:
    u = q.popleft()
    for v in adjacences[u]:
        if v not in dist:
            dist[v] = dist[u] + 1
            q.append(v)
print("Cote 1 joueur : connexe =", len(dist) == 24, "| diametre =", max(dist.values()))
print("  repartition des distances :", dict(sorted(Counter(dist.values()).items())))

# BFS jeu complet : (row, col), 576 sommets
dep2 = ((1, 2, 3, 4), (1, 2, 3, 4))
dist2 = {dep2: 0}
q2 = deque([dep2])
while q2:
    (ur, uc) = q2.popleft()
    d_cur = dist2[(ur, uc)]
    voisins_row = [(swap_valeurs_adjacentes(ur, k), uc) for k in (1, 2, 3)]
    voisins_col = [(ur, swap_valeurs_adjacentes(uc, k)) for k in (1, 2, 3)]
    for v in voisins_row + voisins_col:
        if v not in dist2:
            dist2[v] = d_cur + 1
            q2.append(v)
print()
print("Jeu complet : connexe =", len(dist2) == 576, "| diametre =", max(dist2.values()))
print("  repartition :", dict(sorted(Counter(dist2.values()).items())))
print("  aretes = 576 x 6 / 2 =", 576 * 6 // 2, "-- a comparer aux 1728 murs simples du 1.3")


Cote 1 joueur : connexe = True | diametre = 6
  repartition des distances : {0: 1, 1: 3, 2: 5, 3: 6, 4: 5, 5: 3, 6: 1}

Jeu complet : connexe = True | diametre = 12
  repartition : {0: 1, 1: 6, 2: 19, 3: 42, 4: 71, 5: 96, 6: 106, 7: 96, 8: 71, 9: 42, 10: 19, 11: 6, 12: 1}
  aretes = 576 x 6 / 2 = 1728 -- a comparer aux 1728 murs simples du 1.3


**Lecture de la sortie committée** : le graphe des chambres est **connexe**, de diamètre **6 par côté** et **12 pour le jeu complet** — le pire cas étant l'ordre exactement renversé des deux tables ($6 + 6$ inversions).

Deux régularités lisibles dans la sortie :

- la distribution des distances d'un côté, $\{0{:}1, 1{:}3, 2{:}5, 3{:}6, 4{:}5, 5{:}3, 6{:}1\}$, est **symétrique** : le mot le plus long (l'ordre renversé) est unique, comme le plus court (l'identité) ;
- au niveau du jeu, chaque chambre a **6 voisins** (3 facettes par côté), et le compte d'arêtes $576 \times 6 / 2 = 1728$ coïncide **exactement** avec le nombre de murs simples comptés au §1.3 — chaque mur simple **est** une arête du graphe, chaque arête **est** la traversée d'un mur. Le pont entre le comptage de strates (§1) et la connectivité (§2) est un double-comptage propre.

Conséquence pour la série : le discret $\{1,2,3,4\}$ de GT-3 n'était pas un artefact de commodité — c'est un **transversal** des chambres (un point par chambre), et la connexité ci-dessus garantit que la grammaire des swaps adjacents suffit à relier tout l'univers strict.

## 3. Traverser un mur : chambre → mur → chambre

### 3.1 Un cas complet, matrices à l'appui

Prenons le **Dilemme du Prisonnier** dans l'encodage ordinal de GT-21 — Ligne $(3,1,4,2)$, Colonne $(3,4,1,2)$ — et suivons le swap de valeurs adjacentes $R(3,4)$ (Ligne échange les cases portant 3 et 4). Ce swap traverse la facette où les deux cases porteuses deviennent égales : le mur $r_{11} = r_{21}$.

In [7]:
# === Section 3.1 : traverser le mur r11=r21 depuis le Dilemme du Prisonnier ===

def afficher_jeu(nom, row, col):
    print(nom)
    print(f"  Ligne    | {row[0]:>2}  {row[1]:<2}|   Colonne | {col[0]:>2}  {col[1]:<2}|")
    print(f"           | {row[2]:>2}  {row[3]:<2}|           | {col[2]:>2}  {col[3]:<2}|")
    print()

def faire_le_tie(t, k):
    """Coalescence des valeurs k et k+1 -> mur canonique (make_tie du 4.1, cote seul)."""
    return canonique(tuple(k if v == k + 1 else v for v in t))

# La chambre de depart : le PD (encodage GT-21)
pd_row, pd_col = (3, 1, 4, 2), (3, 4, 1, 2)
afficher_jeu("CHAMBRE A -- Dilemme du Prisonnier (strict)", pd_row, pd_col)

# Le mur : coalescence des valeurs 3 et 4 de Ligne (cases r11 et r21)
mur_row = faire_le_tie(pd_row, 3)
afficher_jeu("MUR (codim 1) -- Ligne indifferent r11 = r21", mur_row, pd_col)

# La chambre d'arrivee : resolution du tie dans l'autre sens
chambre_B_row = swap_valeurs_adjacentes(pd_row, 3)
afficher_jeu("CHAMBRE B -- apres traversee (swap R(3,4))", chambre_B_row, pd_col)

print("Verification : mur =", mur_row)
print("  briser_le_tie(mur) redonne les deux faces :", briser_le_tie(mur_row))
print("  chambre A :", canonique(pd_row), " chambre B :", canonique(chambre_B_row))


CHAMBRE A -- Dilemme du Prisonnier (strict)
  Ligne    |  3  1 |   Colonne |  3  4 |
           |  4  2 |           |  1  2 |

MUR (codim 1) -- Ligne indifferent r11 = r21
  Ligne    |  3  1 |   Colonne |  3  4 |
           |  3  2 |           |  1  2 |

CHAMBRE B -- apres traversee (swap R(3,4))
  Ligne    |  4  1 |   Colonne |  3  4 |
           |  3  2 |           |  1  2 |

Verification : mur = (3, 1, 3, 2)
  briser_le_tie(mur) redonne les deux faces : ((3, 1, 4, 2), (4, 1, 3, 2))
  chambre A : (3, 1, 4, 2)  chambre B : (4, 1, 3, 2)


**Lecture de la sortie committée** : le triplet complet **chambre A → mur → chambre B** est exhibé avec les trois matrices — l'acceptation de l'exercice 2 est satisfaite *et démontrée* :

- **chambre A** : le PD strict, où $r_{21} = 4 > r_{11} = 3$ (Ligne gagne plus à dévier) ;
- **mur** : le jeu à égalité où $r_{11} = r_{21}$ — Ligne est **exactement indifférent** entre ses deux stratégies quand Colonne joue à gauche ; c'est le point de bascule où la défection cesse d'être strictement meilleure *en première colonne* ;
- **chambre B** : l'ordre renversé de la paire $(3,4)$ — le swap $R(3,4)$ de GT-21.

La dernière ligne vérifie mécaniquement la cohérence : le `briser_le_tie` du §2.1 appliqué au mur redonne bien **les deux** chambres — A et B — comme les deux faces de la cloison. Le swap de GT-21, redéfini géométriquement, **est** la traversée de ce mur : rien d'autre.

*Fenêtre pédagogique* : sur le mur, l'ensemble des meilleures réponses de Ligne en première colonne **grossit** (de 1 à 2 choix), puis redevient singulier de l'autre côté — avec l'autre choix. Un mur est exactement le lieu où la structure des meilleures réponses **change de cardinal** — la version locale du « mur habité » de GT-21.

### 3.2 La grammaire des swaps relue en longueurs de Coxeter

GT-21 étudiait **six** swaps par côté : $R(a,b)$ pour toutes les paires de valeurs. La géométrie les sépare en deux familles :

- $|a-b| = 1$ (valeurs **adjacentes**) : le swap traverse **un seul mur** — une facette ;
- $|a-b| \geq 2$ : les valeurs sont séparées par d'autres — le swap est un **raccourci** qui enjambe plusieurs murs.

La longueur exacte (nombre de traversées de facettes) d'une transposition de valeurs à distance $d$ est $2d-1$.

In [8]:
# === Section 3.2 : les swaps GT-21 en longueurs de Coxeter ===

print("Swap  |  traversees de murs  |  statut")
print("-" * 50)
for (a, b) in [(1, 2), (2, 3), (3, 4), (1, 3), (2, 4), (1, 4)]:
    d = abs(a - b)
    statut = "facette (1 mur)" if d == 1 else f"raccourci ({2*d-1} murs enjambes)"
    print(f"R({a},{b})  |         {2*d-1}            |  {statut}")

# Verification : R(1,3) = R(1,2) puis R(2,3) puis R(1,2)
def appliquer_swap(t, a, b):
    return tuple(b if v == a else (a if v == b else v) for v in t)

t0 = (1, 2, 3, 4)
direct = appliquer_swap(t0, 1, 3)
compose = appliquer_swap(appliquer_swap(appliquer_swap(t0, 1, 2), 2, 3), 1, 2)
print()
print("R(1,3) applique a (1,2,3,4)      :", direct)
print("R(1,2) puis R(2,3) puis R(1,2)   :", compose)
print("Egalite (le raccourci = 3 facettes) :", direct == compose)


Swap  |  traversees de murs  |  statut
--------------------------------------------------
R(1,2)  |         1            |  facette (1 mur)
R(2,3)  |         1            |  facette (1 mur)
R(3,4)  |         1            |  facette (1 mur)
R(1,3)  |         3            |  raccourci (3 murs enjambes)
R(2,4)  |         3            |  raccourci (3 murs enjambes)
R(1,4)  |         5            |  raccourci (5 murs enjambes)

R(1,3) applique a (1,2,3,4)      : (3, 2, 1, 4)
R(1,2) puis R(2,3) puis R(1,2)   : (3, 2, 1, 4)
Egalite (le raccourci = 3 facettes) : True


**Lecture de la sortie committée** : la grammaire de GT-21 se recompose géométriquement.

- les trois swaps **adjacents** $R(1,2)$, $R(2,3)$, $R(3,4)$ sont les **générateurs** : chacun traverse exactement un mur (une facette). Avec leurs homologues Colonne, ils engendrent tout le graphe des 576 chambres (§2.2) ;
- les trois **raccourcis** $R(1,3)$, $R(2,4)$, $R(1,4)$ enjambent respectivement 3, 3 et 5 murs. La décomposition est vérifiée dans la sortie : $R(1,3) = R(1,2) \circ R(2,3) \circ R(1,2)$ — le raccourci et la cascade de facettes aboutissent au **même** 4-uplet, mais la cascade *rencontre* les murs intermédiaires (des jeux à égalités où la structure des meilleures réponses peut basculer), quand le raccourci les saute.

C'est la réponse géométrique à une question laissée ouverte par GT-21 : pourquoi certains swaps « ne comptent-ils » pas (préservation des équilibres) ? Parce qu'un swap n'agit sur les équilibres qu'à travers les murs qu'il traverse qui sont **effectivement habités** (au sens meilleures réponses de GT-21). Les facettes rencontrées par la cascade sont autant d'occasions de bascule ; le raccourci n'échantillonne que l'état final.

## 4. Make tie / break tie : les deux opérations de Bruns-Kimmich

### 4.1 Les opérations, implémentées

La description géométrique s'accompagne de deux opérations duuales (rapportées de Bruns-Kimmich, implémentées ici) :

- **make tie** : depuis une chambre, rapprocher deux valeurs adjacentes jusqu'à l'égalité — on **pose** le jeu sur un mur (codim +1) ;
- **break tie** : depuis un mur, séparer l'ex æquo dans un sens ou dans l'autre — on **retombe** dans l'une des deux chambres adjacentes (codim −1).

Ce sont exactement les briques des §2-3, promues au rang d'API.

In [9]:
# === Section 4.1 : make_tie / break_tie, operations duales ===

def make_tie(jeu, cote, k):
    """Coalesce les valeurs k et k+1 du cote donne -> jeu sur un mur (codim +1)."""
    row, col = jeu
    t = faire_le_tie(row, k) if cote == "ligne" else faire_le_tie(col, k)
    return (t, col) if cote == "ligne" else (row, t)

def break_tie(jeu, cote, sens):
    """Separe le premier ex aequo du cote donne, ordre 'haut' ou 'bas' -> chambre (codim -1)."""
    row, col = jeu
    t = row if cote == "ligne" else col
    c = Counter(t)
    v = [k for k, m in c.items() if m == 2]
    if not v:
        return jeu  # pas d'ex aequo : operation identite
    i, j = sorted(p for p in range(4) if t[p] == v[0])
    l = list(t)
    l[i], l[j] = (v[0], v[0] + 1) if sens == "haut" else (v[0] + 1, v[0])
    t2 = canonique(l)
    return (t2, col) if cote == "ligne" else (row, t2)

# Demonstration : aller-retour complet depuis le PD
pd = ((3, 1, 4, 2), (3, 4, 1, 2))
mur = make_tie(pd, "ligne", 3)
retour_haut = break_tie(mur, "ligne", "haut")
retour_bas = break_tie(mur, "ligne", "bas")
print("Chambre (PD)        :", pd)
print("make_tie(ligne, 3)  :", mur, " (codim", codim(mur[0]) + codim(mur[1]), ")")
print("break_tie(haut)     :", retour_haut, "-> retrouve le PD :", retour_haut == pd)
print("break_tie(bas)      :", retour_bas, "-> l'autre chambre (chambre B du 3.1)")
print()
print("Le mur porte bien 2 sorties distinctes :", retour_haut != retour_bas)


Chambre (PD)        : ((3, 1, 4, 2), (3, 4, 1, 2))
make_tie(ligne, 3)  : ((3, 1, 3, 2), (3, 4, 1, 2))  (codim 1 )
break_tie(haut)     : ((3, 1, 4, 2), (3, 4, 1, 2)) -> retrouve le PD : True
break_tie(bas)      : ((4, 1, 3, 2), (3, 4, 1, 2)) -> l'autre chambre (chambre B du 3.1)

Le mur porte bien 2 sorties distinctes : True


**Lecture de la sortie committée** : l'aller-retour est **mécaniquement vérifié**. `make_tie` fait perdre une dimension (le jeu quitte l'ouvert des chambres pour la cloison), `break_tie` la rend — dans **un des deux sens** : la sortie `haut` retrouve le PD d'origine, la sortie `bas` atterrit dans la chambre B du §3.1. Les deux sorties sont distinctes : c'est la matérialisation calculée du « plusieurs prolongements sont possibles » de la thèse botanique.

Un jeu à égalités n'est donc **pas** une spécification incomplète d'un jeu strict : c'est un objet dont le **type** même est différent — un point de l'architecture, avec ses deux faces — et dont le prolongement exige un choix supplémentaire (un sens). Toute la différence entre *assouplir une définition* et *habiter le bord*.

### 4.2 Trois archétypes au bout des murs

Bruns-Kimmich (rapporté) décrivent les jeux 2×2 primitifs comme engendrés depuis des murs par *make/break tie*, sous trois archétypes : **indépendance**, **coordination**, **échange**. Nos implémentations permettent d'en donner des **instanciations calculées** — la définition exacte des archétypes dans la source relève de la dette de sources (§Sources) ; ce qui suit instancie chaque nom par un mur de géométrie naturelle, obtenu par `make_tie` et vérifié par `break_tie`.

In [10]:
# === Section 4.2 : trois instanciations (RAPPORTEES, cf. Sources) ===

print("Archetype       | mur obtenu (Ligne | Colonne)            | geometrie du tie")
print("-" * 90)

# Independance : mur vertical Ligne (r11 = r21) -- Ligne indifferent entre ses
# deux strategies quand Colonne joue a gauche
base_ind = ((1, 3, 2, 4), (1, 3, 2, 4))
mur_ind = make_tie(base_ind, "ligne", 1)
print(f"Independance    | {str(mur_ind[0]):18s} | {str(mur_ind[1]):18s} | vertical (r11 = r21)")

# Coordination : double mur anti-diagonal (r12 = r21 chez les deux joueurs)
base_co = ((1, 2, 3, 4), (1, 2, 3, 4))
mur_co = make_tie(make_tie(base_co, "ligne", 2), "colonne", 2)
print(f"Coordination    | {str(mur_co[0]):18s} | {str(mur_co[1]):18s} | anti-diagonal x2, codim {codim(mur_co[0]) + codim(mur_co[1])}")

# Echange : mur horizontal Ligne (r11 = r12) -- le gain de Ligne en haut ne
# depend pas de la colonne de Colonne
base_ech = ((1, 2, 3, 4), (2, 1, 4, 3))
mur_ech = make_tie(base_ech, "ligne", 1)
print(f"Echange         | {str(mur_ech[0]):18s} | {str(mur_ech[1]):18s} | horizontal (r11 = r12)")

print()
print("Chaque mur se brise dans 2 sens -> chaque archetype borde 2 familles de jeux stricts.")
print("(Instanciations de notre cru ; la definition source des archetypes : dette, cf. Sources.)")


Archetype       | mur obtenu (Ligne | Colonne)            | geometrie du tie
------------------------------------------------------------------------------------------
Independance    | (1, 2, 1, 3)       | (1, 3, 2, 4)       | vertical (r11 = r21)
Coordination    | (1, 2, 2, 3)       | (1, 2, 2, 3)       | anti-diagonal x2, codim 2
Echange         | (1, 1, 2, 3)       | (2, 1, 4, 3)       | horizontal (r11 = r12)

Chaque mur se brise dans 2 sens -> chaque archetype borde 2 familles de jeux stricts.
(Instanciations de notre cru ; la definition source des archetypes : dette, cf. Sources.)


**Lecture de la sortie committée** : chaque archétype est matérialisé par un jeu mur obtenu **par calcul** depuis un jeu strict :

- **indépendance** — mur *vertical* de Ligne ($r_{11} = r_{21}$) : Ligne exactement indifférent entre ses deux stratégies face à la colonne gauche ; brisé vers le haut ou le bas, ce mur fabrique les jeux où Ligne a une préférence stricte — c'est le voisinage des décisions *unilatérales* ;
- **coordination** — double mur *anti-diagonal* (codim 2, une **arête** de l'architecture) : les deux joueurs lisent leur duel sur la même paire de cases ; brisé dans le même sens des deux côtés → coordination pure, en sens opposés → conflit — le voisinage des jeux de rencontre ;
- **échange** — mur *horizontal* de Ligne ($r_{11} = r_{12}$) : le gain de Ligne en haut ne dépend pas de l'action de Colonne ; brisé, il fabrique les jeux où l'issue dépend du croisement des deux décisions — le voisinage des marchandages.

Ces instanciations **calculées** bornent honnêtement ce que ce notebook établit : les opérations, les comptes, les incidences. La généalogie complète des 2×2 par archétypes — les trois primitives de Bruns-Kimmich — est **rapportée**, non dérivée : elle exige la source primaire (§Sources, dette n°4).

## 5. L'espace des 576 jeux et ses générateurs

La première moitié a décrit l'espace et ses strates ; celle-ci pose la question qui reste : *à quelle distance de swap sont deux jeux, et par quels chemins les relie-t-on ?* La distance se mesure en murs traversés — ce que la §2.2 appelait une facette, et que la §3.1 traversait sur un cas complet.

Deux questions d'analyse d'algorithme s'y ajoutent, qui structurent toute cette moitié :

1. **Le générateur propose** — un parcours en largeur (BFS) explore l'espace niveau par niveau et produit un plus court chemin candidat.
2. **Le certificat garantit** — un module compagnon en Lean (`game_theory_lean/Swaps/Basic.lean`) certifie qu'un chemin proposé est bien formé et mène bien du départ à l'arrivée ; sur le cas témoin Dilemme → Chicken, il certifie aussi qu'aucun chemin plus court n'existe. Le générateur Python et le vérificateur Lean restent deux artefacts distincts, chacun à sa place.

Toutes les mesures portent sur l'espace complet des 576 jeux stricts — le quotient par renommage des stratégies (144 classes) est une dette rapportée, jamais affirmée ici.


In [11]:
# Convention GameTheory-3 / GameTheory-21 : une table porte les rangs 1-4
# des quatre cellules dans l'ordre (haut-gauche, haut-droit, bas-gauche,
# bas-droit). Un jeu est le couple (table Ligne, table Colonne).
PERMS = list(permutations((1, 2, 3, 4)))
JEUX = [(r, c) for r in PERMS for c in PERMS]   # 576 jeux stricts

# Le generateur elementaire de cette moitie est celui du 2.2, defini une seule
# fois dans ce carnet : vu des valeurs c'est une relabellisation k <-> k+1, vu
# des cellules c'est la traversee du mur ou ces deux rangs coincident.
swap_adjacent = swap_valeurs_adjacentes

def voisins(g):
    """Les six voisins d'un jeu : trois generateurs par joueur."""
    r, c = g
    return ([(swap_adjacent(r, k), c) for k in (1, 2, 3)] +
            [(r, swap_adjacent(c, k)) for k in (1, 2, 3)])

# Encodages ordinaux canoniques (identiques a GameTheory-3 et GT-21)
CLASSIC = {
    "Dilemme":       ((3, 1, 4, 2), (3, 4, 1, 2)),
    "Chicken":       ((3, 2, 4, 1), (3, 4, 2, 1)),
    "Deadlock":      ((2, 1, 4, 3), (2, 4, 1, 3)),
    "Chasse au cerf": ((4, 1, 3, 2), (4, 3, 1, 2)),
    "Harmonie":      ((4, 3, 2, 1), (4, 2, 3, 1)),
    "Pennies":       ((4, 1, 2, 3), (1, 4, 3, 2)),
}

def bfs(depart):
    """Parcours en largeur : distances et parents depuis un jeu de depart."""
    dist = {depart: 0}
    parent = {depart: None}
    file = deque([depart])
    while file:
        u = file.popleft()
        for v in voisins(u):
            if v not in dist:
                dist[v] = dist[u] + 1
                parent[v] = u
                file.append(v)
    return dist, parent

print(f"Espace : {len(JEUX)} jeux stricts, {len(voisins(JEUX[0]))} generateurs par jeu")
print("generateur partage avec le 2.2 :", swap_adjacent is swap_valeurs_adjacentes)


Espace : 576 jeux stricts, 6 generateurs par jeu
generateur partage avec le 2.2 : True


## 6. La distance d'un jeu à l'autre

La **distance de swap** entre deux jeux est la longueur du plus court chemin de générateurs qui relie l'un à l'autre — le nombre minimal de murs traversés, au sens du §3.1. C'est la distance de graphe sur les 576 sommets, et le parcours en largeur la mesure exactement : le BFS découvre les jeux par sphères croissantes, et la sphère où un jeu apparaît pour la première fois est sa distance au point de départ.

Lançons le BFS depuis le Dilemme du prisonnier — le jeu le plus commenté de la théorie — et demandons les distances aux cinq autres archétypes canoniques.

In [12]:
DIST_PD, PARENT_PD = bfs(CLASSIC["Dilemme"])

print("jeux atteints depuis le Dilemme :", len(DIST_PD), "/ 576")
print("excentricité du Dilemme :", max(DIST_PD.values()))
print()
print("Distances depuis le Dilemme :")
for nom, g in CLASSIC.items():
    print(f"  Dilemme -> {nom:16s} distance = {DIST_PD[g]}")

jeux atteints depuis le Dilemme : 576 / 576
excentricité du Dilemme : 12

Distances depuis le Dilemme :
  Dilemme -> Dilemme          distance = 0
  Dilemme -> Chicken          distance = 2
  Dilemme -> Deadlock         distance = 2
  Dilemme -> Chasse au cerf   distance = 2
  Dilemme -> Harmonie         distance = 6
  Dilemme -> Pennies          distance = 5


### Interprétation : trois cousins à distance 2, un antagoniste à distance 12

Trois archétypes sont à distance 2 du Dilemme — **Chicken, Deadlock et la chasse au cerf** — et un seul est à distance 12 : son **complément ordinal**, le jeu où chaque rang $r$ est remplacé par $5 - r$ chez les deux joueurs (l'« anti-Dilemme », unique point de la sphère maximale). Entre les deux, Pennies à 5 et Harmonie à 6.

La distance 2 des trois cousins n'est pas un artefact d'encodage : chacun s'obtient du Dilemme en échangeant, **chez chaque joueur**, une paire différente de rangs adjacents :

- **Chicken** : l'ordre des deux *pires* rangs ($P \leftrightarrow S$) — c'est tout ce qui sépare la coopération forcée du Dilemme de l'intimidation réciproque ;
- **Deadlock** : l'ordre des rangs du *milieu* ;
- **la chasse au cerf** : l'ordre des deux *meilleurs* rangs ($T \leftrightarrow R$) — le pari de confiance.

Reconstruisons ces chemins avec la table des parents du BFS.

In [13]:
def nommer_etape(g1, g2):
    """Nom du générateur qui fait passer de g1 à son voisin g2."""
    r1, c1 = g1
    r2, c2 = g2
    for k in (1, 2, 3):
        if swap_adjacent(r1, k) == r2:
            return f"R{k}{k + 1}"
        if swap_adjacent(c1, k) == c2:
            return f"C{k}{k + 1}"
    raise ValueError("pas voisins")

def reconstruire(arrivee, parent):
    """Chemin du départ du BFS jusqu'à l'arrivée, en noms de générateurs."""
    chemin = []
    v = arrivee
    while parent[v] is not None:
        chemin.append(nommer_etape(parent[v], v))
        v = parent[v]
    return list(reversed(chemin))

def complement(g):
    """Le complément ordinal : chaque rang r devient 5 - r chez les deux joueurs."""
    return (tuple(5 - v for v in g[0]), tuple(5 - v for v in g[1]))

for nom in ("Chicken", "Deadlock", "Chasse au cerf", "Pennies", "Harmonie"):
    chemin = reconstruire(CLASSIC[nom], PARENT_PD)
    print(f"Dilemme -> {nom:16s} (distance {DIST_PD[CLASSIC[nom]]:2d}) : "
          f"{' '.join(chemin)}")

anti = complement(CLASSIC["Dilemme"])
print(f"\nanti-Dilemme = {anti}")
print(f"distance = {DIST_PD[anti]} ; jeux à cette distance :",
      sum(1 for d in DIST_PD.values() if d == 12))

Dilemme -> Chicken          (distance  2) : R12 C12
Dilemme -> Deadlock         (distance  2) : R23 C23
Dilemme -> Chasse au cerf   (distance  2) : R34 C34
Dilemme -> Pennies          (distance  5) : R34 R23 C12 C23 C12
Dilemme -> Harmonie         (distance  6) : R12 R34 R23 C12 C34 C23

anti-Dilemme = ((2, 4, 1, 3), (2, 1, 4, 3))
distance = 12 ; jeux à cette distance : 1


### Interprétation : la géographie du Dilemme

Le Dilemme habite un carrefour : ses trois cousins célèbres sont à deux pas, chacun dans une direction cardinale différente (bas, milieu, haut de l'échelle des rangs), et l'unique jeu le plus lointain — à 12 swaps — est son **complément ordinal exact**, celui où chaque cellule qui était un bon rang devient un mauvais. Dans ce jeu antagoniste, la situation du Dilemme est intégralement inversée : ce que la coopération produisait de mieux devient pire. La mesure dit autrement la même chose que le §2.2 : traverser les 12 murs qui séparent un jeu de son complément, c'est renverser toute son échelle de valeurs — et aucun chemin ne peut être plus long, car changer les deux tables complètes exige au plus 6 swaps adjacents chacune (le retournement d'un ordre de 4 éléments).

Le point mérite d'être regardé de plus près : la structure de toutes ces distances est remarquablement régulière, et c'est l'objet du §7.

## 7. La distribution des distances : des sphères emboîtées

Le BFS donne la distance de *chaque* jeu au Dilemme — autrement dit la cardinalité de chaque sphère $S_d = \{G : d(\text{Dilemme}, G) = d\}$ pour $d$ de 0 à 12. Si la géographie du Dilemme était quelconque, ces tailles n'auraient aucune raison d'être régulières. Mesurons.

In [14]:
distribution = Counter(DIST_PD.values())
total = sum(distribution.values())
print("d :", " ".join(f"{d:6d}" for d in range(13)))
print("N :", " ".join(f"{distribution[d]:6d}" for d in range(13)))
print("total :", total)
print()
for d in range(13):
    print(f"  d = {d:2d} | {'#' * (distribution[d] // 2):48s} {distribution[d]:4d}")

symetrie = all(distribution[d] == distribution[12 - d] for d in range(13))
print("\nsymétrie d <-> 12 - d :", symetrie)
print("maximum atteint en d = 6 :", distribution[6], "jeux")

d :      0      1      2      3      4      5      6      7      8      9     10     11     12
N :      1      6     19     42     71     96    106     96     71     42     19      6      1
total : 576

  d =  0 |                                                     1
  d =  1 | ###                                                 6
  d =  2 | #########                                          19
  d =  3 | #####################                              42
  d =  4 | ###################################                71
  d =  5 | ################################################   96
  d =  6 | #####################################################  106
  d =  7 | ################################################   96
  d =  8 | ###################################                71
  d =  9 | #####################                              42
  d = 10 | #########                                          19
  d = 11 | ###                                                 6
  d = 12 |  

### Interprétation : une distribution symétrique et factorisée

La distribution est **parfaitement symétrique** autour de 6, en cloche, avec un unique jeu aux deux extrémités ($d = 0$ le Dilemme lui-même, $d = 12$ son complément) et un plateau de 106 jeux à distance 6. Cette régularité appelle une explication structurelle — et elle en a une, calculable de façon indépendante.

La distribution des longueurs de mots sur les 24 permutations d'un côté (la distribution des nombres d'inversions) est le **vecteur de Mahler du permutoèdre** : $\{1, 3, 5, 6, 5, 3, 1\}$ pour $S_4$. Si la distance d'un jeu à l'autre était *la somme indépendante des distances de chaque côté* — un théorème que le §8 établira par énumération exhaustive — alors la distribution depuis n'importe quel jeu devrait être la **convolution de ce vecteur avec lui-même**. Vérifions la prédiction *avant* d'établir le théorème qui la justifie.

In [15]:
# Distribution des longueurs d'un côté : les nombres d'inversions sur S4.
# La distance d'une permutation a l'identite (en swaps adjacents de valeurs)
# est son nombre d'inversions -- la longueur de Coxeter.
def inversions(t):
    return sum(1 for i in range(4) for j in range(i + 1, 4) if t[i] > t[j])

mahler_s4 = Counter(inversions(p) for p in PERMS)
print("vecteur de Mahler de S4 :", [mahler_s4[k] for k in range(7)],
      "(somme :", sum(mahler_s4.values()), "permutations )")

# Convolution manuelle : u * v
def convol(u, v):
    w = [0] * (len(u) + len(v) - 1)
    for i, a in enumerate(u):
        for j, b in enumerate(v):
            w[i + j] += a * b
    return w

predite = convol([mahler_s4[k] for k in range(7)],
                 [mahler_s4[k] for k in range(7)])
mesuree = [distribution[d] for d in range(13)]
print("convolution (Mahler * Mahler) :", predite)
print("distribution mesurée         :", mesuree)
print("coïncidence exacte :", list(predite) == mesuree)

vecteur de Mahler de S4 : [1, 3, 5, 6, 5, 3, 1] (somme : 24 permutations )
convolution (Mahler * Mahler) : [1, 6, 19, 42, 71, 96, 106, 96, 71, 42, 19, 6, 1]
distribution mesurée         : [1, 6, 19, 42, 71, 96, 106, 96, 71, 42, 19, 6, 1]
coïncidence exacte : True


## 8. Le théorème de décomposition

La coïncidence exacte de la cellule précédente n'est pas un hasard : c'est la trace d'un théorème. Énonçons-le d'abord informellement.

> **Décomposition.** La distance de swap entre deux jeux $G = (r_G, c_G)$ et $H = (r_H, c_H)$ est la **somme des distances de chaque côté** :
> $$d(G, H) = \ell(r_G, r_H) + \ell(c_G, c_H)$$
> où $\ell(a, b)$ est la **longueur de Coxeter** de la relabellisation qui envoie $a$ sur $b$ : le nombre d'inversions de la permutation relative $b \circ a^{-1}$, c'est-à-dire le nombre minimal de transpositions de valeurs adjacentes qui transforment $a$ en $b$.

L'intuition : les générateurs Ligne et Colonne sont *indépendants* — un chemin est un entrelacement d'un mot côté Ligne et d'un mot côté Colonne, et chaque côté se comporte comme le graphe de Cayley de $S_4$ muni des trois transpositions adjacentes, dont la distance est exactement la longueur de Coxeter. Nous n'affirmerons pas cette preuve conceptuelle comme établie (elle est rapportée en dette) — mais nous pouvons faire mieux que la croire : **vérifier l'énoncé sur chacune des 331 776 paires de jeux**, en comparant le BFS (la mesure) à la formule (la prédiction), indépendamment l'une de l'autre.

In [16]:
def coxeter(a, b):
    """Longueur de Coxeter de la relabellisation a -> b :
    nombre d'inversions de la permutation relative b . a^{-1}.
    """
    pos = [0] * 5          # pos[v] = position de la valeur v dans a
    for i, v in enumerate(a):
        pos[v] = i
    pi = [b[pos[v]] for v in (1, 2, 3, 4)]   # permutation relative
    return sum(1 for i in range(4) for j in range(i + 1, 4) if pi[i] > pi[j])

# Exemples chiffrés, un par côté du chemin Dilemme -> Chicken :
print("Ligne : (3,1,4,2) -> (3,2,4,1), coxeter =", coxeter((3, 1, 4, 2), (3, 2, 4, 1)))
print("Colonne : (3,4,1,2) -> (3,4,2,1), coxeter =", coxeter((3, 4, 1, 2), (3, 4, 2, 1)))
print("retournement (1,2,3,4) -> (4,3,2,1), coxeter =", coxeter((1, 2, 3, 4), (4, 3, 2, 1)))
print("identite (1,2,3,4) -> (1,2,3,4), coxeter =", coxeter((1, 2, 3, 4), (1, 2, 3, 4)))

Ligne : (3,1,4,2) -> (3,2,4,1), coxeter = 1
Colonne : (3,4,1,2) -> (3,4,2,1), coxeter = 1
retournement (1,2,3,4) -> (4,3,2,1), coxeter = 6
identite (1,2,3,4) -> (1,2,3,4), coxeter = 0


### 8.1 Vérification exhaustive sur les 331 776 paires

Les exemples ci-dessus portent sur trois paires choisies ; l'énoncé, lui, porte sur **toutes**. La vérification est directe et bornée : pour chacune des 576 sources, un parcours en largeur complet donne les distances mesurées, et la formule de Coxeter donne les distances prédites — 576 × 576 = 331 776 paires, dont aucune ne doit diverger.


In [17]:
import time

# Vérification exhaustive : pour chacune des 576 sources, un BFS complet
# (la mesure), comparée a la formule de Coxeter (la prediction) sur les
# 576 destinations. 576 x 576 = 331 776 paires.
t0 = time.perf_counter()
matrice = {}
coherence = True
for G in JEUX:
    dG, _ = bfs(G)
    matrice[G] = dG
    for H in JEUX:
        formule = coxeter(G[0], H[0]) + coxeter(G[1], H[1])
        if dG[H] != formule:
            coherence = False
            print("CONTRE-EXEMPLE :", G, H, dG[H], "!=", formule)
            break
    if not coherence:
        break
elapsed = time.perf_counter() - t0

paires = sum(len(d) for d in matrice.values())
print(f"paires vérifiées : {paires} / 331776 en {elapsed:.1f}s")
print("décomposition d = coxeter(ligne) + coxeter(colonne) :", coherence)

# Corollaires lus sur la matrice complète :
excentricites = Counter(max(dG.values()) for dG in matrice.values())
print("excentricités de tous les jeux :", dict(excentricites))
spot = matrice[CLASSIC["Dilemme"]][CLASSIC["Chicken"]]
print("spot-check matrice [Dilemme][Chicken] =", spot, "(BFS §6 :", DIST_PD[CLASSIC["Chicken"]], ")")

paires vérifiées : 331776 / 331776 en 5.8s
décomposition d = coxeter(ligne) + coxeter(colonne) : True
excentricités de tous les jeux : {12: 576}
spot-check matrice [Dilemme][Chicken] = 2 (BFS §6 : 2 )


### Interprétation : le théorème établi par énumération

**La décomposition vaut pour chacune des 331 776 paires.** Comme le théorème de la condition de transformation dans GT-3d (vérifié sur les 3 456 paires jeu-générateur), l'énoncé est ici *établi par énumération exhaustive* : l'espace est fini, la vérification est complète, et le résultat se lit sur la sortie. La preuve conceptuelle — indépendance des deux côtés, graphe de Cayley de $S_4$, produit cartésien — est **rapportée en dette** (§Sources) : nous n'en livrons pas la démonstration générale, seulement l'évidence exhaustive sur cet espace.

Trois corollaires immédiats, tous mesurés :

- **le diamètre vaut 12** = deux fois le maximum de la longueur de Coxeter d'un côté (6, le retournement) — cohérent avec la mesure du §2.2 ;
- **chaque jeu a une excentricité de 12** : l'espace est parfaitement isotrope, aucun point n'est plus central qu'un autre ; le Dilemme n'occupe aucune position privilégiée — c'est une propriété de *l'espace*, pas du jeu ;
- **la distribution depuis n'importe quel jeu** est la convolution des vecteurs de Mahler (cellule précédente) — ce qui explique la symétrie parfaite autour de 6.

Un point reste ouvert dans la géographie : si les *distances* sont simples, que dire du **nombre** de plus courts chemins ? C'est l'objet du §9.

## 9. Multiplicités : combien de plus courts chemins

La distance dit *combien de pas* ; elle ne dit pas *par où*. Deux jeux à même distance peuvent être reliés par un nombre très différent de chemins minimaux. Comptons-les en énumérant, pour une cible donnée, tous les chemins qui descendent strictement d'un niveau par étape (tout plus court chemin est une telle descente monotone).

In [18]:
def voisins_nommes(g):
    """Les six voisins d'un jeu, avec le nom de leur générateur."""
    r, c = g
    out = []
    for k in (1, 2, 3):
        out.append(((swap_adjacent(r, k), c), f"R{k}{k + 1}"))
        out.append(((r, swap_adjacent(c, k)), f"C{k}{k + 1}"))
    return out

def plus_courts_chemins(depart, arrivee):
    """Enumere tous les plus courts chemins (descentes monotones de niveaux)."""
    d = bfs(depart)[0]
    n = d[arrivee]
    chemins = [(depart, [])]
    for _ in range(n):
        chemins = [(v, p + [nom]) for g, p in chemins
                   for v, nom in voisins_nommes(g) if d.get(v, 99) == d[g] + 1]
    return [p for g, p in chemins if g == arrivee]

from math import comb
for nom in ("Chicken", "Deadlock", "Chasse au cerf", "Pennies", "Harmonie"):
    cs = plus_courts_chemins(CLASSIC["Dilemme"], CLASSIC[nom])
    print(f"Dilemme -> {nom:16s} distance {DIST_PD[CLASSIC[nom]]:2d} : "
          f"{len(cs):6d} plus courts chemins (ex : {' '.join(cs[0])})")

anti = complement(CLASSIC["Dilemme"])
n_anti = len(plus_courts_chemins(CLASSIC["Dilemme"], anti))
# Interpretation structurelle du compte (mesuree independamment) :
identite = ((1, 2, 3, 4), (1, 2, 3, 4))
retourne_ligne = ((4, 3, 2, 1), (1, 2, 3, 4))
mots_w0 = len(plus_courts_chemins(identite, retourne_ligne))
print(f"\nDilemme -> anti-Dilemme : {n_anti} plus courts chemins")
print("mots réduits du retournement, un côté :", mots_w0)
print(f"entrelacements C(12,6) x 16 x 16 = {comb(12, 6)} x {mots_w0} x {mots_w0} =",
      comb(12, 6) * mots_w0 ** 2)

Dilemme -> Chicken          distance  2 :      2 plus courts chemins (ex : R12 C12)
Dilemme -> Deadlock         distance  2 :      2 plus courts chemins (ex : R23 C23)
Dilemme -> Chasse au cerf   distance  2 :      2 plus courts chemins (ex : R34 C34)
Dilemme -> Pennies          distance  5 :     20 plus courts chemins (ex : C12 C23 C12 R34 R23)
Dilemme -> Harmonie         distance  6 :     80 plus courts chemins (ex : R12 C12 R34 R23 C34 C23)



Dilemme -> anti-Dilemme : 236544 plus courts chemins
mots réduits du retournement, un côté : 16
entrelacements C(12,6) x 16 x 16 = 924 x 16 x 16 = 236544


### Interprétation : de la commutation à l'explosion combinatoire

Le spectre des multiplicités est vaste : **exactement 2** chemins minimaux vers chacun des trois cousins (les deux étapes — une par côté — commutent, donc les deux ordres conviennent, et rien d'autre), **20** vers Pennies, **80** vers l'Harmonie, et **236 544** vers l'anti-Dilemme. Ce dernier compte se factorise exactement : $\binom{12}{6} \times 16 \times 16$ — les 16 *mots réduits* du retournement d'un côté (mesurés par la même énumération, restreinte à un seul côté), au carré pour les deux côtés, entrelacés dans $\binom{12}{6}$ ordres possibles. Le chemin minimal vers l'antagoniste absolu n'est donc pas une rareté : il est abondant, presque *tout chemin glouton qui descend monotone y conduit*.

La leçon structurelle : la distance (§6) est une fonction simple et additive, mais la **géométrie fine** des géodésiques porte l'information combinatoire — le nombre de mots réduits d'un élément de $S_4$ est une donnée non triviale de la théorie des groupes de Coxeter, rapportée en dette comme le reste du vocabulaire. C'est précisément quand les objets deviennent abondants et faciles à générer que la question du *certificat* devient pressante : qui garantit qu'un chemin proposé est correct ? C'est le §10.

## 10. Générateur contre certificat

Récapitulons les deux rôles, qui resteront séparés jusqu'au bout :

- le **générateur** (Python, ce notebook) *explore* : BFS, énumération des chemins, comptages. Il **propose** des chemins minimaux — mais un programme qui s'exécute n'est pas une preuve qu'il calcule ce qu'il prétend calculer ;
- le **certificat** (Lean, `game_theory_lean/Swaps/Basic.lean`, écrit pour ce grain) **garantit** : qu'un chemin est bien formé (chaque étape est l'un des six générateurs — c'est *structurel*, le type des étapes ne contient rien d'autre), qu'il mène bien du départ à l'arrivée (évalué par le noyau Lean, `rfl`), et — sur le cas témoin — qu'aucun chemin plus court n'existe (énumération décidable).

Commençons par le miroir Python du vérificateur, pour voir *ce qui se joue* : un chemin **valide** (il arrive) n'est pas un chemin **minimal** (il arrive le plus vite possible).

In [19]:
def applique_etape(g, e):
    """Miroir Python de appliqueEtape (Swaps/Basic.lean) : applique l'étape nommée e."""
    r, c = g
    k = int(e[1])
    return (swap_adjacent(r, k), c) if e[0] == "R" else (r, swap_adjacent(c, k))

def applique_chemin(g, chemin):
    """Miroir Python de applique : application successive des étapes."""
    for e in chemin:
        g = applique_etape(g, e)
    return g

def chemin_valide(depart, arrivee, chemin):
    """Certificat d'arrivée : le chemin mène bien de depart à arrivee.
    Ne dit RIEN de la minimalité."""
    return applique_chemin(depart, chemin) == arrivee

PD, CHICKEN = CLASSIC["Dilemme"], CLASSIC["Chicken"]
c_minimal = ["R12", "C12"]                    # le chemin du BFS (§6)
c_detour = ["R12", "R23", "R23", "C12"]       # un détour : R23 o R23 = identité

print("chemin minimal  ", c_minimal, ": valide =", chemin_valide(PD, CHICKEN, c_minimal))
print("chemin détour   ", c_detour, ": valide =", chemin_valide(PD, CHICKEN, c_detour))
print()
print("longueurs :", len(c_minimal), "vs", len(c_detour),
      "| distance BFS :", DIST_PD[CHICKEN])

chemin minimal   ['R12', 'C12'] : valide = True
chemin détour    ['R12', 'R23', 'R23', 'C12'] : valide = True

longueurs : 2 vs 4 | distance BFS : 2


### Interprétation : valide n'est pas minimal

Le certificat d'arrivée accepte les **deux** chemins — et il a raison : les deux mènent bien du Dilemme à Chicken (le détour insère un aller-retour $R_{23} \circ R_{23}$, qui s'annule). Mais seul le premier est minimal : la longueur 4 du détour dépasse la distance 2. Un vérificateur d'arrivée est donc *complet mais laxiste* ; discriminer les chemins minimaux exige soit l'exploration exhaustive (ce que fait le BFS), soit une borne inférieure (ce que fait le module Lean sur le cas témoin : aucun chemin de longueur < 2 n'existe, par énumération décidable des cas).

C'est la division du travail que le grain installe : **le générateur produit des candidats abondants** (§9 : des centaines de milliers vers l'anti-Dilemme), **le certificat tranche** — arrivée garantie toujours, minimalité garantie sur le témoin. Voyons le module Lean lui-même, cité aux lignes mesurées du fichier.

## 10-bis. Le module Swaps/Basic.lean, cité aux lignes mesurées

Le module compagnon vit dans `game_theory_lean/Swaps/Basic.lean` (même lake que les preuves Arrow et Shapley). Il est **volontairement sans Mathlib** : tout y est calcul fini décidable sur des listes littérales — les théorèmes se closent par `rfl` (évaluation par le noyau) ou `decide` (énumération décidable), sans tactique sophistiquée ni axiome. La cellule suivante lit le fichier depuis ce clone et imprime chaque définition et chaque théorème à sa ligne réelle — les citations du reste de la section s'appuient sur ces numéros, pas sur une mémoire.

In [20]:
from pathlib import Path

def trouver_swaps_lean():
    """Localise Swaps/Basic.lean depuis le répertoire courant ou ses parents
    (le notebook vit dans GameTheory/, le module dans game_theory_lean/)."""
    for base in [Path.cwd(), *Path.cwd().parents]:
        for rel in ("game_theory_lean/Swaps/Basic.lean",
                    "MyIA.AI.Notebooks/GameTheory/game_theory_lean/Swaps/Basic.lean"):
            cible = base / rel
            if cible.exists():
                return cible
    return None

chemin_module = trouver_swaps_lean()
if chemin_module is None:
    print("module Swaps/Basic.lean introuvable depuis ce répertoire —"
          " cloner le dépôt complet pour la lecture des citations")
else:
    src = chemin_module.read_text(encoding="utf-8")
    lignes = src.splitlines()
    print(f"{chemin_module.name} : {len(lignes)} lignes")
    print()
    for i, l in enumerate(lignes, 1):
        if l.startswith(("def ", "theorem ", "inductive ", "structure ")):
            print(f"  {i:4d}: {l.rstrip()[:100]}")
    print()
    print("occurrences de 'sorry' dans le module :", src.count("sorry"))

Basic.lean : 135 lignes

    34: def Table : Type := List Nat
    37: def Jeu : Type := Table × Table
    46: def swapAdj (t : Table) (k : Nat) : Table :=
    50: inductive Etape where
    56: def Etape.k : Etape → Nat
    62: def Etape.ligne : Etape → Bool
    67: def appliqueEtape (g : Jeu) (e : Etape) : Jeu :=
    71: def applique : Jeu → List Etape → Jeu
    78: def dilemme : Jeu := (([3, 1, 4, 2] : Table), [3, 4, 1, 2])
    81: def chicken : Jeu := (([3, 2, 4, 1] : Table), [3, 4, 2, 1])
    84: def cerf : Jeu := (([4, 1, 3, 2] : Table), [4, 3, 1, 2])
    89: def cheminDilemmeChicken : List Etape := [.R12, .C12]
    95: theorem certificat_chemin : applique dilemme cheminDilemmeChicken = chicken := by rfl
   101: theorem chemin_valide_non_minimal :
   106: theorem aucun_chemin_court :
   120: theorem distance_dilemme_chicken :
   130: def cheminValide (depart arrivee : Jeu) (p : List Etape) : Prop :=
   135: theorem certificat_cerf : cheminValide dilemme cerf [.R34, .C34] := by rfl


### Interprétation : ce que chaque théorème garantit

La lecture du module aligne les garanties, dans l'ordre où le notebook les a rencontrées :

- **`certificat_chemin`** — `applique dilemme cheminDilemmeChicken = chicken`, clos par `rfl` : le chemin `[R12, C12]` proposé par le BFS mène bien du Dilemme à Chicken. C'est le certificat d'arrivée, garanti par évaluation du noyau Lean — pas par la ré-exécution d'un programme Python, mais par la réduction du terme lui-même.
- **`chemin_valide_non_minimal`** — le détour `[R12, R23, R23, C12]` arrive aussi (la cellule miroir l'a mesuré côté Python) : le module Lean l'admet explicitement, pour montrer que le certificat d'arrivée n'exclut pas les détours.
- **`aucun_chemin_court`** — aucun chemin de longueur 0 ou 1 ne relie le Dilemme à Chicken : longueur 0 rejetée par `decide` (les jeux diffèrent), longueur 1 par `cases e <;> decide` — les six générateurs sont énumérés un à un, chacun tranché par décision. C'est la **borne inférieure**.
- **`distance_dilemme_chicken`** — la conjonction : un chemin de longueur 2 existe *et* aucun plus court n'existe. La distance vaut **exactement 2** — énoncé complet, prouvé.

La minimalité générale (pour deux jeux quelconques) reste du côté du générateur : elle exige l'exploration. Le module Lean la certifie sur le témoin parce que l'espace des chemins de longueur < 2 est assez petit pour l'énumération décidable — c'est exactement la frontière où le certificat devient abordable, et le grain la marque explicitement plutôt que de la masquer. Le build du module (`lake build Swaps`) est sans dépendance Mathlib : il se compile en secondes, aucune preuve du lake n'est touchée.

## Exercices

### Exercice 1 — Énumérer les murs par codimension

Reproduisez et étendez le comptage du §1 : énumérez les 75 ordres faibles, comptez-les par codimension, puis **par type de blocs** (tailles décroissantes). Vérifiez que la somme des comptes par type redonne 75, et que chaque type de mur $(2,1,1)$, $(2,2)$, $(3,1)$, $(4)$ est non vide.

**Indice** : la signature de blocs s'obtient avec `Counter` ; la codimension est `4 - len(set(t))`. Pour le compte par type, regroupez par `tuple(sorted(Counter(t).values(), reverse=True))`.

**Étape 1** : énumérer les 75 clés canoniques (§1.1). **Étape 2** : grouper par signature. **Étape 3** : afficher le compte par signature et vérifier la somme.

In [21]:
# Exercice 1 : enumeration des murs par codimension et par type de blocs
# TODO etudiant : reproduire la table du 1.2 (75 ordres faibles, 4 signatures de murs)
# Indice : signature = tuple(sorted(Counter(t).values(), reverse=True))
resultat_ex1 = {}  # TODO etudiant : {signature: nombre d'ordres faibles}
print("Exercice 1 : a completer (compte par type de blocs)")


Exercice 1 : a completer (compte par type de blocs)


### Exercice 2 — Traverser : exhiber un triplet complet

Choisissez un jeu strict qui n'est **pas** le PD (par exemple le jeu de la poule mouillée — vérifiez son encodage ordinal avant de l'utiliser), une de ses facettes, et exhibez le triplet **(chambre A, mur, chambre B)** complet avec les trois matrices, comme au §3.1.

**Indice** : une facette = une paire de valeurs adjacentes $(k, k+1)$ ; le mur s'obtient par `faire_le_tie(table, k)` ; la chambre B par `swap_valeurs_adjacentes(table, k)` ; vérifiez enfin que `briser_le_tie(mur)` contient bien les deux chambres.

**Étape 1** : poser les tables du jeu choisi. **Étape 2** : choisir $k$ et construire le mur et la chambre B. **Étape 3** : vérifier la double incidence avec `briser_le_tie`.

In [22]:
# Exercice 2 : exhiber chambre -> mur -> chambre pour un autre jeu
# TODO etudiant : remplir les trois matrices et la verification
chambre_A = None  # TODO etudiant : (row, col) du jeu strict choisi
mur_ex2 = None    # TODO etudiant : le mur traverse (via faire_le_tie)
chambre_B = None  # TODO etudiant : la chambre d'arrivee (via swap_valeurs_adjacentes)
print("Exercice 2 : a completer (triplet chambre-mur-chambre)")


Exercice 2 : a completer (triplet chambre-mur-chambre)


### Exercice 3 — Make tie / break tie : vers les archétypes

Depuis le jeu **Stag Hunt** de GT-21 (Ligne $(4,1,3,2)$, Colonne $(4,3,1,2)$), construisez par `make_tie` un mur de chaque type : un mur Ligne, un mur Colonne, puis un double mur (codim 2). Pour chacun, appliquez `break_tie` dans les deux sens et observez les deux chambres de sortie — vous venez de cartographier le voisinage d'un jeu de coordination.

**Indice** : le double mur s'obtient en composant deux `make_tie` successifs (comme au §4.2 pour la coordination). Les deux sens de `break_tie` doivent redonner deux jeux **stricts** distincts.

**Étape 1** : construire les trois murs. **Étape 2** : pour chacun, briser dans les deux sens. **Étape 3** : vérifier que chaque paire de sorties est distincte et stricte.

In [23]:
# Exercice 3 : voisinage de Stag Hunt par make_tie / break_tie
# TODO etudiant : les trois murs (ligne, colonne, double) et leurs sorties
murs_ex3 = None     # TODO etudiant : {nom_du_mur: jeu_mur}
sorties_ex3 = None  # TODO etudiant : {nom_du_mur: (sortie_haut, sortie_bas)}
print("Exercice 3 : a completer (voisinage de Stag Hunt)")


Exercice 3 : a completer (voisinage de Stag Hunt)


### Exercice 4 — Distance par sphères, sans reconstruire tout le BFS

La fonction `bfs` du §6 calcule les distances vers *tous* les jeux. Pour une seule cible, c'est du gaspillage — et parfois impossible si l'espace était plus grand. Écrivez `distance_par_spheres(depart, arrivee)` qui n'explore que les sphères successives jusqu'à rencontrer l'arrivée : maintenez l'ensemble des jeux du niveau courant, engendez le niveau suivant par générateurs, arrêtez dès que l'arrivée apparaît, et renvoyez le niveau. La fonction doit renvoyer le même verdict que le BFS complet pour toute cible — testez-la au moins sur Chicken (distance 2) et sur l'anti-Dilemme (distance 12).

In [24]:
def distance_par_spheres(depart, arrivee):
    """Distance de swap par exploration des sphères successives seulement.
    # Etape 1 : niveau courant = [depart], distance = 0
    # Etape 2 : engendrer le niveau suivant par les six générateurs
    # Etape 3 : si l'arrivee y figure, renvoyer la distance ; sinon recommencer
    """
    # TODO etudiant
    return None  # a remplacer

resultat_ex1 = distance_par_spheres(CLASSIC["Dilemme"], CLASSIC["Chicken"])
# Indice : la reponse attendue est petite ; le chemin de cette longueur est
# exhibe au §6 et certifie par Swaps/Basic.lean (distance exacte 2).

### Exercice 5 — Le diamètre depuis une autre source : le chiffre, pas l'impression

Le §8 a mesuré que *tous* les jeux ont une excentricité de 12. Vérifiez-le depuis une source différente du Dilemme : écrivez `excentricite_depuis(source)` (distance maximale de la source à tout autre jeu) et appliquez-la à **Pennies**. Puis identifiez **combien de jeux** sont à distance 12 de Pennies et lequel c'est — la réponse est un couple (chiffre, jeu), pas une impression.

In [25]:
def excentricite_depuis(source):
    """Excentricité d'un jeu : sa distance maximale à tout autre jeu.
    # Etape 1 : BFS complet depuis source
    # Etape 2 : renvoyer le maximum des distances
    """
    # TODO etudiant
    return None  # a remplacer

resultat_ex2 = excentricite_depuis(CLASSIC["Pennies"])
# Indice : le theoreme de decomposition (§8) predit la valeur sans
# aucun calcul de BFS -- la formule de Coxeter suffit. Les deux doivent coïncider.

### Exercice 6 — Le certificat d'un chemin fourni : valide, et minimal ?

On vous fournit un chemin du Dilemme à Chicken : `["R12", "R23", "R23", "C12"]`. Écrivez `certificat(depart, arrivee, chemin)` qui renvoie un **couple de booléens** : (validité, minimalité).

- la **validité** se certifie en appliquant le chemin (§10, miroir Python) et en comparant l'arrivée — c'est ce que `Swaps/Basic.lean` garantit par `rfl` ;
- la **minimalité** exige de comparer la longueur du chemin à la distance réelle — soit par re-BFS (exercice 4), soit par la formule de Coxeter (§8).

Votre verdict sur le chemin fourni doit être `(True, False)` : il arrive, mais en 4 pas au lieu de 2. Puis produisez le certificat complet d'un chemin qui mérite les deux `True`.

In [26]:
chemin_fourni = ["R12", "R23", "R23", "C12"]   # un detour, fourni

def certificat(depart, arrivee, chemin):
    """Verdict d'un chemin : couple (validite, minimalite).
    # Etape 1 : validite -- appliquer le chemin, comparer l'arrivee
    # Etape 2 : minimalite -- la longueur egale-t-elle la distance ?
    # Renvoyer (bool, bool)
    """
    # TODO etudiant
    return None  # a remplacer

resultat_ex3 = certificat(CLASSIC["Dilemme"], CLASSIC["Chicken"], chemin_fourni)
# Indice : attendu (True, False) pour le chemin fourni ; un chemin minimal
# (§6) merite (True, True).

## Conclusion

**Ce que la première moitié établit (tout calculé, tout vérifié dans les sorties) :**

1. l'univers des jeux 2×2 **à rangs** compte $75^2 = 5625$ états, dont les 576 jeux stricts ne sont que les chambres ouvertes (10,2 %) — le reste est l'architecture : 1728 murs simples, 1968 strates de codim 2, jusqu'au jeu totalement plat (codim 6, unique) ;
2. l'incidence est exacte : chaque mur simple sépare **exactement 2 chambres**, chaque chambre a **exactement 3 facettes** par côté — et les 1728 arêtes du graphe des chambres coïncident mur pour mur avec les 1728 strates de codim 1 ;
3. le graphe des chambres est **connexe**, de diamètre 6 par côté (12 pour le jeu) — avec la distribution symétrique des distances $\{1,3,5,6,5,3,1\}$ ;
4. la grammaire des swaps de GT-21 se recompose : les 3 swaps adjacents sont des traversées de facettes (1 mur), les 3 autres des raccourcis (3, 3 et 5 murs enjambés), avec $R(1,3) = R(1,2) \circ R(2,3) \circ R(1,2)$ vérifié ;
5. *make tie* / *break tie* sont implémentées et l'aller-retour est vérifié : un jeu à égalités est un objet **biface**, pas une définition relâchée.

**Ce que la seconde moitié établit :**

- **La distance de swap est mesurable exactement** : BFS depuis n'importe quel jeu, sphères emboîtées. Depuis le Dilemme : Chicken, Deadlock et la chasse au cerf à distance 2 (chacun en échangeant une paire différente de rangs adjacents chez chaque joueur), Pennies à 5, Harmonie à 6, l'anti-Dilemme à 12.
- **Théorème de décomposition (établi par énumération exhaustive, 331 776 paires)** : $d(G, H) = \ell(r_G, r_H) + \ell(c_G, c_H)$, la somme des longueurs de Coxeter des permutations relatives. Corollaires mesurés : diamètre 12, excentricité 12 pour *tous* les jeux (isotropie parfaite), distribution depuis tout jeu = convolution des vecteurs de Mahler de $S_4$.
- **Les multiplicités portent la combinatoire** : de 2 chemins minimaux (commutation des deux côtés) vers chacun des trois cousins à 236 544 = $\binom{12}{6} \times 16^2$ vers l'anti-Dilemme — les géodésiques abondent exactement là où la distance est maximale.
- **Générateur contre certificat** : le BFS Python propose, le module Lean `Swaps/Basic.lean` garantit — arrivée par `rfl`, minimalité par énumération décidable sur le témoin Dilemme → Chicken (distance exactement 2). Le module est sans Mathlib et sans `sorry` ; la minimalité générale reste du côté du générateur, et la frontière est explicitée.
- **Position dans la série** : les chambres et les murs sont **dans ce carnet** (§1-§4) ; les chemins — la métrique de l'espace entier — en sont la seconde moitié (§6-§10). Le maillon manquant (rapporté en dette) : le quotient par renommage et la structure $S_4 \times S_4$ comme *tore*, dont notre espace complet est le dépliage.

**Ce qui reste ouvert** — les cinq dettes de dérivation, à honorer depuis les sources primaires avant toute affirmation publique : le quotient $576 \to 144$ (facteur exact à re-dériver), l'action du groupe $S_4 \times S_4$, le tore à 37 trous, la preuve conceptuelle de la décomposition, la théorie générale des groupes de Coxeter derrière les mots réduits — et la vérification firsthand des trois arXiv. Ce carnet ne les affirme pas : il les porte comme dette (§Sources).

**La phrase à retenir** : *un objet botanique n'est pas un objet plus flou — c'est un objet de dimension inférieure depuis lequel plusieurs prolongements sont possibles ; et la distance entre deux chambres se mesure en murs traversés.*

*Chantier des jeux 2×2 — voir aussi* : [GT-3 Topology2x2](GameTheory-03-Topology2x2-Python.ipynb) (l'espace et la grammaire) · [GT-3c Le Joueur LLM](GameTheory-03c-Le-Joueur-LLM-Python.ipynb) (la table périodique appliquée à un LLM) · [GT-3d Deux-Espèces-de-Flèches](GameTheory-03d-Deux-Especes-de-Fleches-Python.ipynb) (quand un swap préserve les équilibres) · [GT-3e Méta-Actions-Tarifées](GameTheory-03e-Meta-Actions-Tarifees-Python.ipynb) (le capstone).


## Sources et dettes

- **Bruns & Kimmich** — la description géométrique (chambres/murs/strates, make/break tie, archétypes) est **rapportée** de leur formalisation des jeux 2×2, non dérivée de la source primaire ici (dette n°4). Tout ce que le carnet affirme chiffré (75, 5625, 1728, 3, 2, 6, 12) est **calculé localement** et reproductible en ré-exécutant.
- **Robinson & Goforth** — le tableau périodique des jeux 2×2 ; ancêtre Rapoport & Guyer. (L'attribution « Gale et Morris » qui circule dans certains transcripts est une hallucination corrigée.)
- **Vocabulaire** : volontairement minimal — chambres, murs, strates, codimension, facettes, distance de swap, générateur, certificat. Le seul terme de groupe employé est celui **que le carnet calcule** : la longueur de Coxeter d'un swap (§3.2) et la somme de ces longueurs qui donne la distance (§8) ; la théorie générale des groupes de Coxeter reste **rapportée** en dette, jamais affirmée.

**Mesuré dans ce carnet** (chaque chiffre est l'output d'une cellule) : les 75 ordres faibles et les 5 625 jeux à rangs par codimension, l'incidence exacte chambre-mur, le graphe des chambres (connexité, diamètre, distribution), les longueurs de Coxeter des six swaps, l'aller-retour *make tie* / *break tie* ; puis les distances aux archétypes, la distribution complète, la vérification exhaustive de la décomposition (331 776 paires), les excentricités, les multiplicités des plus courts chemins, la factorisation $\binom{12}{6} \times 16^2$, le miroir Python du vérificateur et les citations de `Swaps/Basic.lean` aux lignes réelles.

**Dettes ouvertes (HARD), jamais affirmées** :

1. **Le quotient $576 \to 144$** (classes de jeux à renommage des stratégies près) et la structure $S_4 \times S_4$ comme tore — Robinson et Goforth rapportent cette lecture ; tout ce carnet travaille sur l'espace complet des 576 jeux et n'affirme rien du quotient.
2. **La preuve conceptuelle de la décomposition** (indépendance des côtés, graphe de Cayley, produit cartésien de graphes) — nous en livrons l'énumération exhaustive, pas la démonstration générale.
3. **Le tore à 37 trous** et la topologie du quotient — hors scope, rapporté.
4. **Les références arXiv** (2309.15981, 2102.00053, 1704.02230) — identités vérifiées en première main (API arXiv) : Tohmé & Viglizzo, *A categorical representation of games* · Czechowski & Piliouras, *Poincaré-Bendixson Limit Sets in Multi-Agent Learning* · Hedges, *Coherence for lenses and open games* ; la lecture intégrale des trois reste à faire.
5. **La théorie des mots réduits** (16 mots pour le retournement de $S_4$) — mesurée par énumération ; la théorie générale des groupes de Coxeter qui l'explique est rapportée.

**Module compagnon** : `game_theory_lean/Swaps/Basic.lean` — pur core Lean, sans Mathlib, sans `sorry` ; `lake build Swaps` sans dépendance.

**Encodages** : identiques à GameTheory-3 et GT-3d (tables de rangs 1-4, cellules dans l'ordre haut-gauche, haut-droit, bas-gauche, bas-droit).
